# Demo of the End-to-End Imaging Loop (CLEAN)


[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/processing_functions_tutorials/imaging/demo_imaging_loop.ipynb)


This notebook demonstrates the AstroViper imaging loop, a `tclean`-like major/minor
cycle CLEAN. The loop is driven by the **science-layer processing function**
`image_cube_single_field`, which implements:
- Major/minor cycle iteration control (`IterationController`)
- Visibility-domain residual calculation
- Hogbom deconvolution (C++ minor cycle)
- Convergence visualization

> **API change.** The standalone `run_imaging_loop` from
> `astroviper.core.imaging.imager` has been **removed**. The equivalent
> major/minor-cycle CLEAN now lives in the science layer as
> `astroviper.processing_functions.imaging.image_cube_single_field`, which
> operates on an in-memory processing set (`ps_xdt`) and an image dataset
> (`img_xds`). The synthetic `generate_ms4_with_point_sources` helper and the
> low-level `standard_grid` / `grid2image_spheroid_ms4` entry points have also
> been removed; this notebook uses a small real processing set instead.

## Install Astroviper and Xradio (needed for colab)

In [ ]:
import os
from importlib.metadata import version

try:
    os.system('pip install "xradio[casacore]"')
    os.system("pip install --upgrade astroviper")

    import astroviper

    print("Using astroviper version", version("astroviper"))

except ImportError as exc:
    print(f"Could not import astroviper: {exc}")

## Setup

First, we import the required modules. The CLEAN loop comes from the
`processing_functions.imaging` science layer; the iteration-control and
convergence-plotting helpers live alongside it in
`processing_functions.imaging.utils.iteration_control`.

In [ ]:
import holoviews as hv
import numpy as np
import xarray as xr

hv.extension("bokeh")

# Data access (XRADIO)
from xradio.image import make_empty_sky_image
from xradio.measurement_set import load_processing_set, open_processing_set

# Science-layer CLEAN loop (processing function).
# This replaces the removed standalone `run_imaging_loop`.
from astroviper.processing_functions.imaging.image_cube_single_field import (
    image_cube_single_field,
)

# Iteration control + convergence tooling.
# NOTE: the module path moved to processing_functions.imaging.utils.iteration_control.
from astroviper.processing_functions.imaging.utils.iteration_control import (
    ConvergencePlots,
    IterationController,
    ReturnDict,
    plot_convergence_history,
)

## Load a Processing Set (MSv4)

The old notebook synthesized 4 point sources with the now-removed
`generate_ms4_with_point_sources`. Instead we download a small **real**
processing set: TW Hya ALMA data, split to 5 LSRK channels, 2-pol linear
(`XX`, `YY`). This is the same data used by the high-level
`cube_single_field_imaging` tutorial.

`open_processing_set` is lazy (metadata only), while `load_processing_set` pulls
the visibilities into memory — the science-layer processing function operates on
in-memory `xarray` / `NumPy` data.

In [ ]:
from toolviper.utils.data import download, update

PS_STORE = "twhya_selfcal_5chans_lsrk_compare_weights.ps.zarr"

# Download the small real MSv4 processing set (no-op if already present).
update()
download(file=PS_STORE)

# Open lazily for metadata (phase center, frequency axis); load the data we
# image into memory.
ps_open = open_processing_set(PS_STORE)
ps_xdt = load_processing_set(PS_STORE, data_group_name="base", load_sub_datasets=False)

combined = ps_open.xr_ps.get_combined_field_and_source_xds()
phase_center = combined.FIELD_PHASE_CENTER_DIRECTION.sel(
    field_name=combined.attrs["center_field_name"]
).values
frequency_coords = ps_open.xr_ps.get_freq_axis().values

ms_name = list(ps_xdt.keys())[0]
dims = ps_xdt[ms_name].sizes
print(f"Loaded processing set: {PS_STORE}")
print(f"Measurement set: {ms_name}")
print(
    f"Dimensions: time={dims['time']}, baseline_id={dims['baseline_id']}, "
    f"frequency={dims['frequency']}, polarization={dims['polarization']}"
)
print(f"Polarizations: {list(ps_xdt[ms_name].polarization.values)}")

## Polarization Handling

The TW Hya dataset has 2 polarizations (`XX`, `YY`). The imaging loop grids in
the instrument (correlation) basis and outputs Stokes:
- 2-pol linear (`XX`, `YY`) → Stokes `I`, `Q`
- 2-pol circular (`RR`, `LL`) → Stokes `I`, `V`

The correlation basis is set by the `instrument_polarization_basis` argument
(`"linear"` here), and the Stokes output basis by
`image_params["polarization_coords"]`.

In [ ]:
ms_xds = ps_xdt[ms_name]

print(f"MS polarizations: {list(ms_xds.polarization.values)}")
print(
    f"VISIBILITY shape: {ms_xds['VISIBILITY'].shape}  # (time, baseline_id, frequency, polarization)"
)
print(
    "  -> gridded in the linear correlation basis (XX, YY), output as Stokes ['I', 'Q']"
)

In [ ]:
# Metadata that feeds the image geometry below.
print(f"Frequency axis ({len(frequency_coords)} channels), Hz:")
for i, f in enumerate(frequency_coords):
    print(f"  chan {i}: {f:.6e}")
print(f"\nPhase center (RA, Dec) radians: {phase_center}")
print(f"\nMeasurement-set 'base' data group: {ms_xds.attrs['data_groups']['base']}")

## Configure the Imaging Loop

`image_cube_single_field` takes **three parameter dicts** (matching the
high-level driver):

- **`image_params`** — image geometry and output coordinates: `image_size`,
  `cell_size` (radians; the `l`/RA element is negative by convention),
  `phase_direction`, `frequency_coords`, `polarization_coords` (the Stokes
  output basis), `time_coords`, `fft_padding` and `cpp_gridder`.
- **`imaging_weights_params`** — `"natural"` or `"briggs"` weighting (with the
  Briggs `robust` parameter).
- **`iteration_control_params`** — the CLEAN controls (CASA `tclean` semantics,
  applied **independently per (time, frequency, polarization) plane**):
  `niter` (max minor-cycle components per plane), `nmajor` (max major cycles;
  `-1` = unlimited), `threshold` (Jy, absolute stop), `gain`,
  `primary_beam_limit`, and the cycle knobs `cyclefactor` / `cycleniter` /
  `minpsffraction` / `maxpsffraction`.

In [ ]:
# Image geometry and output coordinates.
image_size = [200, 200]
cell_size = np.array([-0.1, 0.1]) * np.pi / (180 * 3600)  # 0.1 arcsec pixels

image_params = {
    "image_size": image_size,
    "cell_size": cell_size,
    "phase_direction": phase_center,
    "frequency_coords": frequency_coords,
    "polarization_coords": ["I", "Q"],  # Stokes output basis (2-pol linear -> I, Q)
    "time_coords": [0],  # collapse all times into one plane
    "fft_padding": 1.2,
    "cpp_gridder": True,
}

# Weighting scheme.
imaging_weights_params = {
    "weighting": "briggs",
    "robust": 0.5,
    "casa_weighting_implementation": True,
}

# CLEAN iteration control -- capped at 3 major cycles for this demo.
iteration_control_params = {
    "niter": 300,  # max minor-cycle components per plane
    "nmajor": 3,  # max major cycles (demo cap)
    "threshold": 0.001,  # Jy, absolute stop
    "primary_beam_limit": 0.2,
    "gain": 0.1,
    "cyclefactor": 1.5,
    "cycleniter": -1,
    "minpsffraction": 0.05,
    "maxpsffraction": 0.8,
}

print("Imaging parameters configured:")
print(f"  Image: {image_size[0]} x {image_size[1]} pixels")
print(f"  Cell size: {abs(cell_size[0]) * 206265:.3f} arcsec")
print(f"  Max major cycles: {iteration_control_params['nmajor']}")
print(f"  Max iterations/plane: {iteration_control_params['niter']}")
print(f"  Threshold: {iteration_control_params['threshold'] * 1000:.1f} mJy")

## Run the Imaging Loop

We first build an **empty image dataset** (`img_xds`) in the correlation
(instrument) basis the gridder works in (`XX`, `YY`), then hand it, the
processing set, and the three parameter dicts to `image_cube_single_field`.
Internally, per the `IterationController`, this:
1. Computes imaging weights, the PSF (Stokes I only) and the primary beam.
2. For each major cycle: grids the residual visibilities to an image,
   transforms to the Stokes basis, runs Hogbom minor cycles into the model,
   then degrids the model and forms new residual visibilities.
3. Runs a final residual cycle, and (with `restore=True`) a restored image.

It returns the populated `img_xds` (with `SKY_MODEL`, `SKY_RESIDUAL`,
`SKY_RESTORED`, `POINT_SPREAD_FUNCTION`, `PRIMARY_BEAM`), a one-row timing
`DataFrame`, and the per-plane convergence `ReturnDict`.

In [ ]:
# Empty per-chunk image in the correlation (instrument) basis (linear -> XX, YY).
# The science function transforms it to the Stokes output basis internally.
img_xds = make_empty_sky_image(
    phase_center=image_params["phase_direction"],
    image_size=image_params["image_size"],
    cell_size=image_params["cell_size"],
    frequency_coords=image_params["frequency_coords"],
    pol_coords=["XX", "YY"],
    time_coords=image_params["time_coords"],
    do_sky_coords=True,
)

# Run the science-layer CLEAN loop.
img_xds, timing_df, deconvolve_dict = image_cube_single_field(
    ps_xdt,
    img_xds,
    image_params,
    imaging_weights_params,
    iteration_control_params,
    processing_set_data_group_name="base",
    deconvolver="hogbom",
    instrument_polarization_basis="linear",
    single_precision_image=False,
    processing_function_threads=1,
    fft_backend="scipy",
    image_data_variables_keep=[
        "sky_residual",
        "sky_model",
        "point_spread_function",
        "primary_beam",
    ],
    restore=True,
)

print("CLEAN loop complete.")
print(f"Major cycles run: {int(timing_df['n_major_cycles'].iloc[0])}")

## Examine Results

Let's look at the final model and residual images.

In [ ]:
print(f"Image dims: {dict(img_xds.sizes)}")
print(
    f"SKY_MODEL shape:    {img_xds['SKY_MODEL'].shape}  # (time, frequency, polarization, l, m)"
)
print(f"SKY_RESIDUAL shape: {img_xds['SKY_RESIDUAL'].shape}")
print()

n_major = int(timing_df["n_major_cycles"].iloc[0])
n_planes = len(deconvolve_dict.data)
total_iters = sum(
    (
        sum(v["iter_done"])
        if isinstance(v.get("iter_done"), list)
        else v.get("iter_done", 0)
    )
    for v in deconvolve_dict.data.values()
)
print(f"Major cycles run: {n_major}")
print(f"(time, pol, chan) planes cleaned: {n_planes}")
print(f"Total minor-cycle iterations (summed over planes): {total_iters}")

# Per-plane stop reason (each plane carries its own CASA-style stop code).
first_key = list(deconvolve_dict.data.keys())[0]
print(
    f"Example plane {first_key}: {deconvolve_dict.data[first_key].get('stop_description')}"
)

In [ ]:
# Plot the Stokes I model, residual and restored images (middle channel).
chan = 2
model_I = img_xds["SKY_MODEL"].isel(time=0, frequency=chan, polarization=0).values
residual_I = img_xds["SKY_RESIDUAL"].isel(time=0, frequency=chan, polarization=0).values
restored_I = img_xds["SKY_RESTORED"].isel(time=0, frequency=chan, polarization=0).values

img_model = hv.Image(model_I).opts(
    tools=["hover"],
    title=f"SKY_MODEL (Stokes I, chan {chan}) — flux {np.sum(model_I):.3f} Jy",
    width=400,
    height=400,
    colorbar=True,
    cmap="viridis",
)
img_resid = hv.Image(residual_I).opts(
    tools=["hover"],
    title=f"SKY_RESIDUAL (Stokes I) — peak {np.max(np.abs(residual_I)):.4f} Jy",
    width=400,
    height=400,
    colorbar=True,
    cmap="viridis",
)
img_restored = hv.Image(restored_I).opts(
    tools=["hover"],
    title="SKY_RESTORED (model*beam + residual)",
    width=400,
    height=400,
    colorbar=True,
    cmap="viridis",
)

hv.Layout([img_model, img_resid, img_restored]).cols(3)

In [ ]:
# With real data there are no known synthetic source positions to compare
# against (the old source-recovery table used generate_ms4_with_point_sources).
# Instead, list the brightest CLEAN components the minor cycle placed in the model.
model_I = img_xds["SKY_MODEL"].isel(time=0, frequency=chan, polarization=0).values
top = np.argsort(model_I.ravel())[::-1][:5]

print(f"Brightest CLEAN model components (Stokes I, channel {chan}):")
print(f"{'pixel (m, l)':<18}{'flux (Jy)':<12}")
print("-" * 30)
for idx in top:
    m, l = np.unravel_index(idx, model_I.shape)
    print(f"({m:3d}, {l:3d})        {model_I[m, l]:.4f}")
print(f"\nTotal Stokes I model flux (channel {chan}): {np.sum(model_I):.3f} Jy")

## Convergence Visualization

The third return value, `deconvolve_dict`, is a `ReturnDict` of per-plane
convergence history (keyed by `(time, pol, chan)`). We visualize it with
`plot_convergence_history` or the `ConvergencePlots` class.

The convergence plot shows:
- **Peak Residual** (blue, left y-axis): how the peak residual decreases over iterations
- **Model Flux** (red, right y-axis): how the total model flux grows as flux is cleaned

In [ ]:
# Inspect the ReturnDict contents to see what was tracked.
print("ReturnDict (per-plane deconvolution statistics):")
print(f"  Number of planes: {len(deconvolve_dict.data)}")
print(f"  Keys (time, pol, chan): {list(deconvolve_dict.data.keys())}")

# Show first plane details.
first_key = list(deconvolve_dict.data.keys())[0]
print(f"\nFirst plane (key={first_key}):")
for field, value in deconvolve_dict.data[first_key].items():
    if isinstance(value, list) and len(value) > 3:
        print(
            f"  {field}: [{value[0]}, {value[1]}, ..., {value[-1]}] ({len(value)} values)"
        )
    else:
        print(f"  {field}: {value}")

In [ ]:
# Interactive convergence plot using HoloViews.
# Dual y-axis plot with selectors for Stokes parameter and channel.
convergence_plot = plot_convergence_history(
    deconvolve_dict,
    time=0,
    stokes="I",
    chan=0,
    width=800,
    height=400,
)

convergence_plot

In [ ]:
# Alternative: use the ConvergencePlots class directly for more control.
plotter = ConvergencePlots(deconvolve_dict)
plotter.plot_history(time=0, stokes="I", chan=0)

## Summary

This notebook demonstrated:

1. **Loading a real processing set** (TW Hya, 5 channels, 2-pol linear) with
   XRADIO's `open_processing_set` / `load_processing_set`.
2. **Native 2-pol support** — gridding in the `XX`, `YY` correlation basis and
   producing Stokes `I`, `Q` directly.
3. **Running the CLEAN loop** with the science-layer
   `image_cube_single_field` (the replacement for the removed
   `run_imaging_loop`).
4. **Results inspection** — `SKY_MODEL`, `SKY_RESIDUAL` and `SKY_RESTORED`
   images written into `img_xds`.
5. **Convergence visualization** from the returned `ReturnDict` via
   `plot_convergence_history` / `ConvergencePlots`.

The loop implements a complete `tclean`-like workflow: visibility-domain
residuals, major/minor cycle architecture, adaptive per-plane cycle
thresholding, comprehensive convergence tracking, and native 2-pol / 4-pol
support.

In [ ]:
# Final summary
model_flux_I = float(img_xds["SKY_MODEL"].isel(polarization=0).sum())
resid_peak_I = float(np.abs(img_xds["SKY_RESIDUAL"].isel(polarization=0)).max())

print("=" * 60)
print("IMAGING LOOP SUMMARY")
print("=" * 60)
print(
    f"Data: TW Hya, {img_xds.sizes['frequency']} channels, "
    f"Stokes {list(img_xds.polarization.values)}"
)
print(f"Total model flux (Stokes I, all channels): {model_flux_I:.3f} Jy")
print(f"Peak residual (Stokes I): {resid_peak_I:.4f} Jy")
print(f"Major cycles: {int(timing_df['n_major_cycles'].iloc[0])}")
print(f"Planes cleaned: {len(deconvolve_dict.data)}")
print("=" * 60)

## Note: low-level gridding comparison removed

Earlier versions of this notebook ended with a comparison between
`grid2image_spheroid_ms4` and a manual `standard_grid_numpy_wrap_input_checked`
call. Those low-level gridding entry points (and the surrounding
`standard_grid` module) have been **removed** from AstroViper.

Gridding now happens *inside* the science-layer processing functions —
`make_point_spread_function_single_field` and
`residual_cycle_cube_single_field`, which `image_cube_single_field` calls
internally — using the templated C++ prolate-spheroidal gridder (selected via
`image_params["cpp_gridder"]`). For a direct look at PSF gridding, see the
`make_psf_demo` notebook in this directory.